In [1]:
# ==========================================
# 📝 Cell 1: 检查 GPU
# ==========================================
!nvidia-smi
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")

Wed Feb 25 22:57:28 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# ==========================================
# 📝 Cell 2: 安装依赖（一次性全装）
# ==========================================
print("📦 安装所有依赖...")

!pip install -q transformers==4.35.2 tokenizers==0.15.0
!pip install -q hydra-core omegaconf
!pip install -q supervision opencv-python pillow matplotlib
!pip install -q addict yapf timm scipy terminaltables pycocotools

print("\n✅ 依赖安装完成！")

# 验证（但不要导入 transformers！）
import sys
print(f"Python: {sys.version}")

📦 安装所有依赖...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.5/123.5 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.9/7.9 MB 113.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 119.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 50.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.2.3 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.35.2 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 6.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.9/41.9 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 217.4/217.4 kB 15.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.8/46.8 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 256.2/25

In [3]:
# ==========================================
# 📝 Cell 3: 克隆仓库
# ==========================================
%cd /content

# 删除旧的（如果存在）
!rm -rf Grounded-SAM-2 segment-anything-2

# 克隆新的
!git clone https://github.com/IDEA-Research/Grounded-SAM-2.git
!git clone https://github.com/facebookresearch/segment-anything-2.git

print("✅ 仓库克隆完成！")

/content
Cloning into 'Grounded-SAM-2'...
remote: Enumerating objects: 2065, done.
remote: Counting objects: 100% (19/19), done.
remote: Compressing objects: 100% (14/14), done.
remote: Total 2065 (delta 11), reused 5 (delta 5), pack-reused 2046 (from 2)
Receiving objects: 100% (2065/2065), 230.08 MiB | 42.25 MiB/s, done.
Resolving deltas: 100% (769/769), done.
Cloning into 'segment-anything-2'...
remote: Enumerating objects: 1070, done.
remote: Total 1070 (delta 0), reused 0 (delta 0), pack-reused 1070 (from 1)
Receiving objects: 100% (1070/1070), 128.11 MiB | 39.56 MiB/s, done.
Resolving deltas: 100% (380/380), done.
✅ 仓库克隆完成！


In [4]:
# ==========================================
# 📝 Cell 4: 安装 SAM2
# ==========================================
%cd /content/segment-anything-2
!pip install -q -e ".[demo]"
print("✅ SAM2 安装完成！")

/content/segment-anything-2
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 3.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Building editable for SAM-2 (pyproject.toml) ... done
✅ SAM2 安装完成！


In [5]:
# ==========================================
# 📝 Cell 5: 安装 Grounding DINO
# ==========================================
%cd /content/Grounded-SAM-2
!pip install --no-build-isolation -e grounding_dino
print("✅ Grounding DINO 安装完成！")

/content/Grounded-SAM-2
Obtaining file:///content/Grounded-SAM-2/grounding_dino
  Preparing metadata (setup.py) ... done
  Running setup.py develop for groundingdino
✅ Grounding DINO 安装完成！


In [6]:
# ==========================================
# 📝 Cell 6: 下载模型权重
# ==========================================
import os

# 创建目录
os.makedirs("/content/Grounded-SAM-2/checkpoints", exist_ok=True)
os.makedirs("/content/Grounded-SAM-2/gdino_checkpoints", exist_ok=True)

# 下载 SAM2
%cd /content/Grounded-SAM-2/checkpoints
!wget -q --show-progress https://dl.fbaipublicfiles.com/segment_anything_2/072824/sam2_hiera_large.pt

# 下载 Grounding DINO
%cd /content/Grounded-SAM-2/gdino_checkpoints
!wget -q --show-progress https://github.com/IDEA-Research/GroundingDINO/releases/download/v0.1.0-alpha/groundingdino_swint_ogc.pth

print("\n✅ 模型权重下载完成！")

/content/Grounded-SAM-2/checkpoints
sam2_hiera_large.pt 100%[===================>] 856.35M   205MB/s    in 4.2s    
/content/Grounded-SAM-2/gdino_checkpoints
groundingdino_swint 100%[===================>] 661.85M   134MB/s    in 4.2s    

✅ 模型权重下载完成！


In [7]:
# ==========================================
# 📝 Cell: 重启后直接运行这个！
# ==========================================
import os
import sys
import torch

# 1. 重新进入目录 (重启后会回到根目录，必须重新进入)
%cd /content/Grounded-SAM-2

# 2. 强制把当前目录加入 Python 搜索路径
# 这一步是为了防止 Python 找不到刚才安装的包
sys.path.append(os.getcwd())

# 3. 尝试导入
print("🚀 正在尝试导入 GroundingDINO...")
try:
    from groundingdino.util.inference import load_model
    print("✅ 成功导入 groundingdino！")
except ImportError:
    # 如果还是不行，尝试添加子目录
    sys.path.append(os.path.join(os.getcwd(), "grounding_dino"))
    from groundingdino.util.inference import load_model
    print("✅ 通过子目录导入成功！")

# 4. 加载模型
config_file = "grounding_dino/groundingdino/config/GroundingDINO_SwinT_OGC.py"
checkpoint_path = "groundingdino_swint_ogc.pth"

if not os.path.exists(checkpoint_path):
    !wget -q https://github.com/IDEA-Research/GroundingDINO/releases/download/v0.1.0-alpha/groundingdino_swint_ogc.pth

print("🚀 正在初始化模型...")
grounding_dino_model = load_model(config_file, checkpoint_path)
device = "cuda" if torch.cuda.is_available() else "cpu"
grounding_dino_model = grounding_dino_model.to(device)

print(f"🎉 模型加载完毕！变量 'grounding_dino_model' 已就绪。")
print("👉 现在去运行你的【批量处理】代码吧！")

/content/Grounded-SAM-2
🚀 正在尝试导入 GroundingDINO...


✅ 通过子目录导入成功！
🚀 正在初始化模型...


final text_encoder_type: bert-base-uncased


The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

🎉 模型加载完毕！变量 'grounding_dino_model' 已就绪。
👉 现在去运行你的【批量处理】代码吧！


In [9]:
# ==========================================
# 📝 Cell: 挂载 Google Drive
# ==========================================
from google.colab import drive
import os

# 检查是否已经挂载，避免重复挂载提示
if not os.path.exists('/content/drive'):
    print("正在挂载 Google Drive...")
    drive.mount('/content/drive')
else:
    print("✅ Google Drive 已经挂载！")

正在挂载 Google Drive...
Mounted at /content/drive


In [10]:
# ==========================================
# 📝 Cell: 强制修复路径并加载模型
# ==========================================
import sys
import os
import torch

# --- 🚑 修复步骤 1: 强制添加路径 ---
# 告诉 Python 去哪里找 groundingdino
cwd = os.getcwd()
print(f"当前工作目录: {cwd}")

# 假设你的目录结构是标准的 Grounded-SAM-2
# 我们需要把 grounding_dino 这个子文件夹加入系统路径
gd_path = os.path.join(cwd, "grounding_dino")

if gd_path not in sys.path:
    sys.path.append(gd_path)
    print(f"🔧 已手动添加路径: {gd_path}")

# --- 🚑 修复步骤 2: 再次尝试导入 ---
try:
    from groundingdino.util.inference import load_model
    print("✅ 成功导入 groundingdino 模块！")
except ImportError as e:
    print(f"❌ 依然导入失败: {e}")
    print("正在尝试备用路径...")
    # 备用方案：有时候库直接装在根目录下
    if "/content/Grounded-SAM-2" not in sys.path:
        sys.path.append("/content/Grounded-SAM-2")
    try:
        from groundingdino.util.inference import load_model
        print("✅ 备用路径导入成功！")
    except ImportError:
        print("❌ 彻底失败。请查看下方的【终极解决方案】。")
        raise  # 抛出错误停止运行

# --- 步骤 3: 加载模型 (如果导入成功) ---
config_file = os.path.join(gd_path, "groundingdino/config/GroundingDINO_SwinT_OGC.py")
checkpoint_path = "groundingdino_swint_ogc.pth"

if not os.path.exists(checkpoint_path):
    print("📥 下载权重文件...")
    !wget -q https://github.com/IDEA-Research/GroundingDINO/releases/download/v0.1.0-alpha/groundingdino_swint_ogc.pth

print("🚀 正在初始化模型...")
try:
    grounding_dino_model = load_model(config_file, checkpoint_path)
    print("🎉 模型加载成功！变量 'grounding_dino_model' 已准备好。")
    print("👉 现在去运行你的【批量处理】代码吧！")
except Exception as e:
    print(f"❌ 模型初始化错误: {e}")
    print("请检查 config_file 路径是否正确:", config_file)

当前工作目录: /content/Grounded-SAM-2
✅ 成功导入 groundingdino 模块！
🚀 正在初始化模型...
final text_encoder_type: bert-base-uncased
🎉 模型加载成功！变量 'grounding_dino_model' 已准备好。
👉 现在去运行你的【批量处理】代码吧！


In [11]:
# ==========================================
# 📝 Cell: 初始化 SAM 2 模型 (缺的就是这一步！)
# ==========================================
import os
import torch
from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor

# 1. 确保在正确的目录
%cd /content/Grounded-SAM-2

# 2. 定义模型参数 (使用 Large 模型效果最好)
sam2_checkpoint = "./checkpoints/sam2.1_hiera_large.pt"
model_cfg = "configs/sam2.1/sam2.1_hiera_l.yaml"

# 3. 检查权重文件，没有就下载
if not os.path.exists("./checkpoints"):
    os.makedirs("./checkpoints")

if not os.path.exists(sam2_checkpoint):
    print(f"📥 正在下载 SAM 2 权重: {sam2_checkpoint} ...")
    !wget -q -P ./checkpoints https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_large.pt
    print("✅ 下载完成")

# 4. 加载模型并创建 predictor 变量
print("🚀 正在初始化 SAM 2 模型...")

try:
    device = "cuda" if torch.cuda.is_available() else "cpu"

    # 构建模型
    sam2_model = build_sam2(model_cfg, sam2_checkpoint, device=device)

    # 创建预测器 (这就是你报错缺少的那个变量！)
    sam2_predictor = SAM2ImagePredictor(sam2_model)

    print(f"🎉 SAM 2 模型加载成功！(运行在 {device} 上)")
    print("👉 变量 'sam2_predictor' 已创建。")
    print("👉 现在再次运行你的【批量处理】代码，应该就能成功分割了！")

except Exception as e:
    print(f"❌ 加载失败: {e}")
    print("可能是配置文件路径不对。请查看左侧文件栏，确认 'sam2_configs' 或 'configs' 文件夹里的文件名。")
    # 备用方案：如果是旧版本代码，可能配置路径不同
    if "No such file" in str(e):
        print("💡 尝试备用配置路径...")
        try:
            model_cfg = "sam2_configs/sam2_hiera_l.yaml" # 旧版路径
            sam2_model = build_sam2(model_cfg, sam2_checkpoint, device=device)
            sam2_predictor = SAM2ImagePredictor(sam2_model)
            print("🎉 备用路径加载成功！")
        except:
            print("❌ 备用路径也失败了，请检查文件结构。")

/content/Grounded-SAM-2
📥 正在下载 SAM 2 权重: ./checkpoints/sam2.1_hiera_large.pt ...
✅ 下载完成
🚀 正在初始化 SAM 2 模型...
🎉 SAM 2 模型加载成功！(运行在 cuda 上)
👉 变量 'sam2_predictor' 已创建。
👉 现在再次运行你的【批量处理】代码，应该就能成功分割了！


In [16]:
import os
import numpy as np
import torch
import torchvision
from PIL import Image
import matplotlib.pyplot as plt
import warnings
import torchvision.transforms as T

warnings.filterwarnings('ignore')

# ... (路径设置) ...
DRIVE_IMAGE_FOLDER = "/content/drive/MyDrive/sam2/notebooks/images"
DRIVE_RESULTS_FOLDER = "/content/drive/MyDrive/sam2/notebooks/images/results"
os.makedirs(DRIVE_RESULTS_FOLDER, exist_ok=True)

# ... (提示词) ...
IMAGE_PROMPTS = {
    "bai.jpg": "stuffed toy.",
    "bunny.png": "plush bunny.",
    "cars.jpg": "car.",
    "groceries.jpg": "paper bag.",
    "truck.jpg": "pickup truck."
}

# ==========================================
# 🔧 0. 修复 load_image (关键修改：只传一个参数)
# ==========================================
def load_image(image_path):
    """
    使用标准 torchvision 加载图像
    """
    # 1. 加载图片
    image_pil = Image.open(image_path).convert("RGB")

    # 2. 定义标准转换
    # Grounding DINO 预训练模型通常需要 Resize 到 800 左右，并进行标准化
    transform = T.Compose([
        T.Resize((800, 800)), # 简单起见强制缩放，或者使用 T.Resize(800) 保持比例
        T.ToTensor(),
        T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])

    # 3. 转换
    image_source = np.asarray(image_pil)

    # --- 关键修复：这里只传 image_pil，不要传 None ---
    image_transformed = transform(image_pil)

    return image_source, image_transformed

def box_convert(boxes, in_fmt, out_fmt):
    """
    手动实现坐标转换: 'cxcywh' -> 'xyxy'
    """
    if in_fmt == 'cxcywh' and out_fmt == 'xyxy':
        cx, cy, w, h = boxes.unbind(-1)
        b = [(cx - 0.5 * w), (cy - 0.5 * h),
             (cx + 0.5 * w), (cy + 0.5 * h)]
        return torch.stack(b, dim=-1)
    return boxes

# ==========================================
# 🔧 1. 检测参数
# ==========================================
DETECTION_PARAMS = {
    "default": {
        "box_threshold": 0.30,
        "text_threshold": 0.25,
        "iou_threshold": 0.5
    },
    "cars.jpg": {
        "box_threshold": 0.20,
        "text_threshold": 0.20,
        "iou_threshold": 0.5
    }
}

# ==========================================
# 🔧 2. 可视化函数
# ==========================================
def visualize_results(image_source, masks, boxes, phrases, save_path):
    plt.figure(figsize=(15, 10))
    plt.imshow(image_source)

    # 绘制 mask
    if masks is not None and len(masks) > 0:
        for mask in masks:
            color = np.random.random(3)
            mask_2d = mask.squeeze() if isinstance(mask, np.ndarray) else mask.cpu().numpy().squeeze()

            if mask_2d.ndim == 2:
                h, w = mask_2d.shape
                mask_rgba = np.zeros((h, w, 4))
                mask_rgba[mask_2d > 0, :3] = color
                mask_rgba[mask_2d > 0, 3] = 0.5
                plt.imshow(mask_rgba)

    # 绘制边界框
    for box, phrase in zip(boxes, phrases):
        x1, y1, x2, y2 = box
        w_box = x2 - x1
        h_box = y2 - y1

        plt.gca().add_patch(
            plt.Rectangle((x1, y1), w_box, h_box,
                         fill=False, edgecolor='red', linewidth=3)
        )
        plt.text(x1, max(y1-10, 0), phrase,
                color='white', fontsize=12, weight='bold',
                bbox=dict(facecolor='red', alpha=0.8, pad=2))

    plt.axis('off')
    plt.tight_layout()
    plt.savefig(save_path, bbox_inches='tight', dpi=150, pad_inches=0)
    plt.close()
    print(f"  💾 图片已保存: {save_path}")

# ==========================================
# 🎯 主循环
# ==========================================
print("\n🚀 开始批量处理（修复 Compose 参数错误）\n")

for image_name, text_prompt in IMAGE_PROMPTS.items():
    print(f"\n{'='*70}")
    print(f"🖼️  处理: {image_name}")

    image_path = os.path.join(DRIVE_IMAGE_FOLDER, image_name)
    if not os.path.exists(image_path):
        print(f"❌ 文件不存在: {image_path}")
        continue

    # 1. 加载图片
    try:
        image_source, image_tensor = load_image(image_path)
    except Exception as e:
        print(f"  ❌ 加载错误: {e}")
        import traceback
        traceback.print_exc()
        continue

    params = DETECTION_PARAMS.get(image_name, DETECTION_PARAMS["default"])
    print(f"  ⚙️ 使用参数: {params}")

    # 2. Grounding DINO 检测
    try:
        # 假设 predict 函数已定义
        boxes_norm, logits, phrases = predict(
            model=grounding_dino_model,
            image=image_tensor,
            caption=text_prompt,
            box_threshold=params["box_threshold"],
            text_threshold=params["text_threshold"],
            device="cuda"
        )
        print(f"  📊 检测到 {len(boxes_norm)} 个目标")
    except NameError:
        print("  ❌ 错误: 'predict' 或 'grounding_dino_model' 未定义。")
        break
    except Exception as e:
        print(f"  ❌ 检测模型错误: {e}")
        continue

    if len(boxes_norm) == 0:
        print("  ⚠️ 未检测到任何目标")
        continue


    # ============================================================
    # 🔧 新增步骤: NMS (非极大值抑制) 去除重复框
    # ============================================================

    # 1. 将 cxcywh (中心点+宽高) 转换为 xyxy (左上右下)，因为 NMS 需要 xyxy 格式
    # 注意：这里的 boxes_norm 是归一化的 (0-1)，但不影响 NMS 计算 IoU
    boxes_norm_xyxy = box_convert(boxes=boxes_norm, in_fmt="cxcywh", out_fmt="xyxy")

    # 2. 执行 NMS
    # iou_threshold=0.5 表示如果两个框重叠超过 50%，就删掉分数低的那个
    # 建议值：0.5 到 0.7 之间。对于排列紧密的袋子，0.5 比较安全。
    nms_idx = torchvision.ops.nms(boxes_norm_xyxy, logits, params["iou_threshold"])

    # 3. 根据 NMS 的结果筛选框、分数和标签
    boxes_norm = boxes_norm[nms_idx]
    logits = logits[nms_idx]
    phrases = [phrases[i] for i in nms_idx] # phrases 是列表，需要列表推导式筛选

    print(f"  ✂️ NMS 过滤后: {len(boxes_norm)} 个目标")


    # 3. 坐标转换 (cxcywh -> xyxy)
    # 注意：Grounding DINO 输出的是归一化的 cxcywh
    # 我们需要将其转换为像素坐标的 xyxy 给 SAM2 和画图使用
    h, w, _ = image_source.shape

    # 先转格式 cxcywh -> xyxy (归一化状态)
    boxes_norm_xyxy = box_convert(boxes=boxes_norm, in_fmt="cxcywh", out_fmt="xyxy")

    # 再转像素 (归一化 -> 像素)
    boxes_pixel = (boxes_norm_xyxy * torch.Tensor([w, h, w, h])).numpy()

    # 4. SAM2 分割
    try:
        sam2_predictor.set_image(image_source)
        masks, scores_sam, _ = sam2_predictor.predict(
            point_coords=None,
            point_labels=None,
            box=boxes_pixel,
            multimask_output=False,
        )
        print(f"  ✅ 分割完成")
    except NameError:
        print("  ❌ 错误: 'sam2_predictor' 未定义。")
        break
    except Exception as e:
        print(f"  ❌ SAM2 分割错误: {e}")
        masks = []

    # 5. 可视化
    result_path = os.path.join(DRIVE_RESULTS_FOLDER, f"fixed_{image_name}")
    visualize_results(image_source, masks, boxes_pixel, phrases, result_path)

print("\n🎉 所有任务完成！")


🚀 开始批量处理（修复 Compose 参数错误）


🖼️  处理: bai.jpg
  ⚙️ 使用参数: {'box_threshold': 0.3, 'text_threshold': 0.25, 'iou_threshold': 0.5}
  📊 检测到 1 个目标
  ✂️ NMS 过滤后: 1 个目标
  ✅ 分割完成
  💾 图片已保存: /content/drive/MyDrive/sam2/notebooks/images/results/fixed_bai.jpg

🖼️  处理: bunny.png
  ⚙️ 使用参数: {'box_threshold': 0.3, 'text_threshold': 0.25, 'iou_threshold': 0.5}
  📊 检测到 1 个目标
  ✂️ NMS 过滤后: 1 个目标
  ✅ 分割完成
  💾 图片已保存: /content/drive/MyDrive/sam2/notebooks/images/results/fixed_bunny.png

🖼️  处理: cars.jpg
  ⚙️ 使用参数: {'box_threshold': 0.2, 'text_threshold': 0.2, 'iou_threshold': 0.5}
  📊 检测到 2 个目标
  ✂️ NMS 过滤后: 2 个目标
  ✅ 分割完成
  💾 图片已保存: /content/drive/MyDrive/sam2/notebooks/images/results/fixed_cars.jpg

🖼️  处理: groceries.jpg
  ⚙️ 使用参数: {'box_threshold': 0.3, 'text_threshold': 0.25, 'iou_threshold': 0.5}
  📊 检测到 8 个目标
  ✂️ NMS 过滤后: 4 个目标
  ✅ 分割完成
  💾 图片已保存: /content/drive/MyDrive/sam2/notebooks/images/results/fixed_groceries.jpg

🖼️  处理: truck.jpg
  ⚙️ 使用参数: {'box_threshold': 0.3, 'text_threshold': 0.25, 'iou_t